# Simple RAG Pipeline — Hands-On Demo

A minimal but honest Retrieval-Augmented Generation system, built with
**LangChain** and **Gemini 3.5 Flash Lite**.

We answer questions over three synthetic company policy documents:

| Document | Covers |
| --- | --- |
| `returns_policy.md` | Return windows, refunds, restocking fees |
| `shipping_policy.md` | Delivery speeds, pickup, shipping restrictions |
| `warranty_policy.md` | Manufacturer warranty, protection plans, claims |

The documents deliberately **overlap** — returns and warranties both talk about
30-day windows — so retrieval has to actually discriminate rather than guess.

### The pipeline

```text
INGESTION   load -> chunk -> embed -> index
RETRIEVAL   question -> embed -> search -> assemble context
GENERATION  prompt -> LLM -> parse -> validate -> answer
```

Everything downstream of retrieval is where most demos get lazy. We won't —
section 9 verifies the model's citations before anything reaches the user.

---
## 0. Environment check

Confirm the notebook is running against the project virtual environment. If the
kernel is wrong, select **Python (RAG Workshop)** from the kernel picker.

In [1]:
import sys
from pathlib import Path

print(f"Python     : {sys.version.split()[0]}")
print(f"Executable : {sys.executable}")

import langchain
import langchain_google_genai

print(f"LangChain  : {langchain.__version__}")

DATA_DIR = Path("data").resolve()
assert DATA_DIR.is_dir(), f"Missing data directory: {DATA_DIR}"
print(f"Data dir   : {DATA_DIR}")
print(f"Documents  : {sorted(p.name for p in DATA_DIR.glob('*.md'))}")

Python     : 3.11.13
Executable : /Users/a0g09q3/Documents/Advanced Agentic AI recordings/rag_workshop/.venv/bin/python
LangChain  : 1.3.17
Data dir   : /Users/a0g09q3/Documents/Advanced Agentic AI recordings/rag_workshop/data
Documents  : ['returns_policy.md', 'shipping_policy.md', 'warranty_policy.md']


### API key

`gemini-3.5-flash-lite` is called through Google AI Studio. Grab a free key at
<https://aistudio.google.com/apikey>.

The key is read from the `GOOGLE_API_KEY` environment variable, from a local
`.env` file, or prompted for securely with `getpass` — so it never lands in the
notebook output or in git.

> **On a restricted network?** If the embedding model is already cached locally
> but Hugging Face is unreachable, set `HF_HUB_OFFLINE=1` before starting
> Jupyter. The cell below does this automatically when a cache is present.

In [2]:
import getpass
import os

# Use the cached embedding model rather than hitting a blocked network.
cache = Path.home() / ".cache/huggingface/hub"
if (cache / "models--sentence-transformers--all-MiniLM-L6-v2").is_dir():
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    print("Embedding model found in cache — running Hugging Face offline")

# Load a local .env if one exists (it is gitignored).
env_file = Path(".env")
if env_file.is_file():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

# langchain-google-genai reads GOOGLE_API_KEY; accept GEMINI_API_KEY too.
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API key: ")

print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

Embedding model found in cache — running Hugging Face offline
API key loaded: True


---
# Part 1 — Ingestion pipeline

## 1. Load the documents

`DirectoryLoader` walks a folder and produces a `Document` per file. Each
`Document` carries `page_content` plus a `metadata` dict.

That metadata is not decoration. It is what lets you cite sources, filter by
access level, and debug bad answers later. Populate it at load time — it is
painful to backfill once chunks are in the index.

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    str(DATA_DIR),
    glob="*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False,
)
documents = loader.load()

# Enrich metadata: a readable title beats an absolute file path in a citation.
for doc in documents:
    filename = Path(doc.metadata["source"]).name
    doc.metadata["filename"] = filename
    doc.metadata["title"] = filename.replace("_", " ").replace(".md", "").title()

print(f"Loaded {len(documents)} documents\n")
for doc in documents:
    print(f"  {doc.metadata['title']:<28} {len(doc.page_content):>6,} chars")

/var/folders/21/x1jqmvg96kl_ylqn71c2ny5c0000gp/T/ipykernel_87570/657764398.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
/Users/a0g09q3/Documents/Advanced Agentic AI recordings/rag_workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 3 documents

  Shipping Policy               2,919 chars
  Returns Policy                2,775 chars
  Warranty Policy               2,964 chars


## 2. Chunk the documents

Why chunk at all? Two reasons:

1. **Precision.** Embedding a whole document averages every topic in it into a
   single vector, which ends up close to nothing in particular.
2. **Context budget.** You cannot paste an entire corpus into a prompt.

`RecursiveCharacterTextSplitter` tries separators in order — paragraphs, then
lines, then sentences, then words — so it breaks at the most natural boundary
that fits the size limit.

**Overlap** matters: a fact that straddles a chunk boundary would otherwise be
cut in half. A 10–20% overlap is the usual sweet spot.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 600
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
    length_function=len,
)
chunks = splitter.split_documents(documents)

# Stable ids make it possible to trace an answer back to an exact chunk.
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"c{index:03d}"

sizes = [len(c.page_content) for c in chunks]
print(f"{len(documents)} documents -> {len(chunks)} chunks")
print(f"size: min={min(sizes)}  mean={sum(sizes) // len(sizes)}  max={max(sizes)}")

3 documents -> 23 chunks
size: min=114  mean=374  max=586


In [5]:
# Inspect one chunk. Always eyeball your chunks before trusting the index.
sample = chunks[3]
print(f"chunk_id : {sample.metadata['chunk_id']}")
print(f"source   : {sample.metadata['title']}")
print(f"length   : {len(sample.page_content)} chars")
print("-" * 68)
print(sample.page_content)

chunk_id : c003
source   : Shipping Policy
length   : 298 chars
--------------------------------------------------------------------
## Membership benefits

Members receive free standard shipping with no minimum basket size, and free
same-day delivery on baskets over $35. Membership costs $98 per year or $12.95
per month.

Members also receive early access to seasonal sale events, typically four
hours before the general public.


## 3. Embed the chunks

An embedding maps text to a vector such that *similar meaning* lands in a
*similar place*. This is what lets a search for "money back" find a passage
about "refunds" with no shared keywords.

We use `all-MiniLM-L6-v2` — small, fast, runs locally, 384 dimensions. It is
downloaded once (~90 MB) and cached.

> Keeping embeddings local while generation is hosted is a common and sensible
> split: the corpus never leaves the machine during indexing.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

# If you are facing issues in bringing up MiniLM Embedding, uncomment following lines to use Gemini Embedding model

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

probe = embeddings.embed_query("How long do I have to return a laptop?")
print(f"Embedding dimensions: {len(probe)}")
print(f"First 5 values: {[round(v, 4) for v in probe[:5]]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19761.83it/s]


Embedding dimensions: 384
First 5 values: [-0.0176, 0.0323, 0.0735, -0.0048, 0.0254]


### Sanity check: does semantic similarity actually work?

Before building an index, prove the embeddings behave. Cosine similarity between
a question and a *relevant* sentence should clearly beat an *irrelevant* one.
If this check fails, nothing downstream can possibly work.

In [7]:
import numpy as np

question = "Can I get my money back on a broken toaster?"
candidates = [
    "Refunds are issued to the original payment method within 3 to 5 days.",
    "Same-day delivery is available within a 15-mile radius of a store.",
]

q_vec = np.array(embeddings.embed_query(question))
for text in candidates:
    score = float(np.dot(q_vec, np.array(embeddings.embed_query(text))))
    print(f"{score:.3f}  {text}")

0.276  Refunds are issued to the original payment method within 3 to 5 days.
0.047  Same-day delivery is available within a 15-mile radius of a store.


## 4. Build the vector index

FAISS stores the vectors and answers nearest-neighbour queries fast. For a few
thousand chunks it is effectively instant and needs no server.

This is the final step of ingestion. Everything so far runs **offline and
ahead of time** — not per question.

In [8]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("faiss_index")

print(f"Indexed {vector_store.index.ntotal} vectors of dim {vector_store.index.d}")
print("Saved to ./faiss_index")

Indexed 23 vectors of dim 384
Saved to ./faiss_index


---
# Part 2 — Retrieval pipeline

## 5. Retrieve relevant chunks

At query time we embed the question with the *same* model and pull the nearest
chunks.

`similarity_search_with_score` also returns an L2 distance. **Lower is better.**
We convert it to an intuitive `relevance` score in roughly 0–1 so the numbers
read the right way round.

In [9]:
TOP_K = 4


def retrieve(question: str, k: int = TOP_K) -> list[dict]:
    """Return the k most relevant chunks, with a readable relevance score."""
    hits = vector_store.similarity_search_with_score(question, k=k)
    return [
        {
            "rank": rank,
            "chunk_id": doc.metadata["chunk_id"],
            "title": doc.metadata["title"],
            "text": doc.page_content,
            "distance": float(distance),
            "relevance": round(1.0 / (1.0 + float(distance)), 3),
        }
        for rank, (doc, distance) in enumerate(hits, start=1)
    ]


for hit in retrieve("How long do I have to return a television?"):
    print(f"{hit['rank']}. [{hit['relevance']:.3f}] {hit['title']} ({hit['chunk_id']})")
    print(f"   {hit['text'][:110].strip()}...\n")

1. [0.511] Returns Policy (c009)
   # Returns and Refunds Policy

Document ID: POL-RET-004
Owner: Customer Experience
Last reviewed: March 2026

#...

2. [0.479] Returns Policy (c014)
   ## Large and heavy items

Furniture, appliances, and televisions larger than 55 inches require a
scheduled pic...

3. [0.478] Returns Policy (c010)
   ## Exceptions to the standard window

The following categories have shorter return windows:

- Consumer electr...

4. [0.460] Warranty Policy (c021)
   For major appliances and televisions over 55 inches, an in-home technician
visit is scheduled instead of depot...



### The overlap test

Ask something that lives in the seam between two documents. Both the returns
policy and the warranty policy mention a 30-day window — for entirely different
things. Watch which chunks surface, and notice this is exactly the kind of
question a naive system gets wrong.

In [10]:
for hit in retrieve("What is covered after 30 days?"):
    print(f"{hit['rank']}. [{hit['relevance']:.3f}] {hit['title']:<30} {hit['chunk_id']}")

1. [0.504] Warranty Policy                c022
2. [0.491] Returns Policy                 c010
3. [0.481] Warranty Policy                c018
4. [0.458] Warranty Policy                c019


## 6. Assemble the context

Retrieved chunks get formatted into a single labelled block. The labels are
load-bearing: `[1]`, `[2]` give the model a vocabulary for citing its sources,
which we verify later.

We also apply a **relevance floor**. If the best hit is weak, the honest move is
to stop and say we don't know, rather than hand the model junk and hope.

In [11]:
RELEVANCE_FLOOR = 0.30


def build_context(hits: list[dict]) -> str:
    """Format retrieved chunks into a numbered, citable context block."""
    return "\n\n".join(
        f"[{hit['rank']}] (source: {hit['title']})\n{hit['text'].strip()}"
        for hit in hits
    )


hits = retrieve("How much does same-day delivery cost?")
context = build_context(hits)

print(f"Top relevance: {hits[0]['relevance']}  (floor: {RELEVANCE_FLOOR})")
print(f"Context: {len(context)} chars from {len(hits)} chunks\n")
print(context[:600] + "\n...")

Top relevance: 0.636  (floor: 0.3)
Context: 1585 chars from 4 chunks

[1] (source: Shipping Policy)
## Delivery speeds

| Service           | Cost                          | Typical arrival     |
| ----------------- | ----------------------------- | ------------------- |
| Standard          | Free over $35, else $5.99     | 3 to 5 business days |
| Express           | $9.99                         | 2 business days     |
| Next day          | $14.99                        | Next business day   |
| Same day delivery | $9.99, or free for members    | Within 4 hours      |

[2] (source: Shipping Policy)
## Delivery areas

Same-day delivery is available within a **1
...


---
# Part 3 — Generation and output processing

## 7. The prompt

The prompt is the contract. Ours does three jobs:

1. **Scopes** the model to the supplied context only
2. **Licenses ignorance** — an explicit escape hatch beats an invented answer
3. **Demands citations**, so claims are traceable

In [12]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a company policy assistant. Answer strictly from the numbered "
        "context passages provided.\n"
        "Rules:\n"
        "1. Use only facts present in the context. Never use outside knowledge.\n"
        "2. If the context does not contain the answer, reply exactly: "
        "I don't know based on the available policy documents.\n"
        "3. Cite the passage numbers you used, e.g. [1] or [2].\n"
        "4. Be concise. Quote exact figures, fees, and timeframes.",
    ),
    ("human", "Context passages:\n{context}\n\nQuestion: {question}"),
])

print(RAG_PROMPT.format(context="[1] (source: Demo)\nSample text.", question="Demo?"))

System: You are a company policy assistant. Answer strictly from the numbered context passages provided.
Rules:
1. Use only facts present in the context. Never use outside knowledge.
2. If the context does not contain the answer, reply exactly: I don't know based on the available policy documents.
3. Cite the passage numbers you used, e.g. [1] or [2].
4. Be concise. Quote exact figures, fees, and timeframes.
Human: Context passages:
[1] (source: Demo)
Sample text.

Question: Demo?


## 8. The generator, as a chain

We wire the pieces together with LCEL, LangChain's pipe syntax. `StrOutputParser`
is the built-in parser that pulls plain text out of the model's message object,
so we never touch `.content` by hand:

```text
prompt | llm | StrOutputParser()
```

> `gemini-3.5-flash-lite` uses fixed sampling defaults, so it ignores
> `temperature`. On models that honour it, set `temperature=0` — creativity is
> a bug in a policy assistant.

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

# LCEL: prompt -> model -> plain string
rag_chain = RAG_PROMPT | llm | StrOutputParser()

print(rag_chain.invoke({
    "context": "[1] (source: Demo)\nThe demo fee is $4.99.",
    "question": "What is the demo fee?",
}))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The demo fee is $4.99 [1].


## 9. Output processing

`StrOutputParser` gave us a clean string. That is *parsing* — but it is not yet
an answer you would ship. Two light checks turn the text into something with
provenance attached:

1. **Extract the citations** the model wrote, keeping only those that point at
   passages we actually supplied. A model asked for four passages will
   occasionally cite `[7]`.
2. **Flag ungrounded answers.** An answer with no citations, from a prompt that
   demanded citations, has earned your suspicion. This is the cheapest
   hallucination smoke alarm available.

No bespoke JSON parser, no fence-stripping regex — just a small amount of
verification on top of LangChain's built-in parser.

In [14]:
import re


def attach_sources(answer: str, hits: list[dict]) -> dict:
    """Verify the model's citations and attach the real source documents."""
    # Pull [1], [2] style markers out of the answer text.
    cited = {int(n) for n in re.findall(r"\[(\d+)\]", answer)}

    # Keep only citations that map to a passage we actually supplied.
    sources = [hit for hit in hits if hit["rank"] in cited]

    return {
        "answer": answer.strip(),
        "sources": [
            {"rank": s["rank"], "title": s["title"],
             "chunk_id": s["chunk_id"], "relevance": s["relevance"]}
            for s in sources
        ],
        "grounded": bool(sources),
    }


demo_hits = [
    {"rank": 1, "title": "Returns Policy", "chunk_id": "c001", "relevance": 0.8},
    {"rank": 2, "title": "Shipping Policy", "chunk_id": "c009", "relevance": 0.6},
]
# Citation [7] was never supplied, so it is silently dropped.
print(attach_sources("Most items may be returned within 90 days [1][7].", demo_hits))

{'answer': 'Most items may be returned within 90 days [1][7].', 'sources': [{'rank': 1, 'title': 'Returns Policy', 'chunk_id': 'c001', 'relevance': 0.8}], 'grounded': True}


## 10. The complete pipeline

Every piece, wired together:

```text
question -> retrieve -> floor check -> context -> chain -> verify -> answer
```

The floor check short-circuits before spending a token on hopeless context.
Refusing to answer is a feature, not a failure.

In [15]:
def answer_question(question: str, k: int = TOP_K, verbose: bool = True) -> dict:
    """Run the full RAG pipeline for a single question."""
    hits = retrieve(question, k=k)

    # Guard: bail out before calling the LLM if retrieval found nothing useful.
    if not hits or hits[0]["relevance"] < RELEVANCE_FLOOR:
        result = {
            "answer": "I don't know based on the available policy documents.",
            "sources": [],
            "grounded": False,
            "refused": True,
        }
    else:
        answer = rag_chain.invoke({
            "context": build_context(hits),
            "question": question,
        })
        result = attach_sources(answer, hits) | {"refused": False}

    result["question"] = question
    result["top_relevance"] = hits[0]["relevance"] if hits else 0.0

    if verbose:
        cited = ", ".join(
            f"{s['title']} ({s['chunk_id']})" for s in result["sources"]
        ) or "none"
        print(f"Q: {question}")
        print(f"A: {result['answer']}")
        print(f"   grounded={result['grounded']}  refused={result['refused']}  "
              f"top_relevance={result['top_relevance']}")
        print(f"   sources: {cited}")

    return result


_ = answer_question("How long do I have to return a drone?")

Q: How long do I have to return a drone?
A: Consumer electronics and drones have a return window of **30 days** [2].
   grounded=True  refused=False  top_relevance=0.486
   sources: Returns Policy (c010)


## 11. Try it out

Four question types, chosen to expose different behaviours:

- **Straightforward** — the fact sits in one chunk
- **Cross-document** — needs returns *and* warranty
- **Out of scope** — the honest answer is "I don't know"
- **Specific figure** — tests whether numbers survive the round trip

In [16]:
demo_questions = [
    "How much does express shipping cost?",
    "What is the difference between a return and a warranty claim?",
    "What is the company policy on parental leave?",
    "What restocking fee applies to large items?",
]

for question in demo_questions:
    answer_question(question)
    print("-" * 76)

Q: How much does express shipping cost?
A: Express shipping costs $9.99 [1].
   grounded=True  refused=False  top_relevance=0.667
   sources: Shipping Policy (c001)
----------------------------------------------------------------------------
Q: What is the difference between a return and a warranty claim?
A: A warranty claim is different from a return in that returns are governed by POL-RET-004 and are time-limited to the return window, whereas warranty claims can be made after the return window has closed [1].
   grounded=True  refused=False  top_relevance=0.693
   sources: Warranty Policy (c017)
----------------------------------------------------------------------------
Q: What is the company policy on parental leave?
A: I don't know based on the available policy documents.
   grounded=False  refused=False  top_relevance=0.405
   sources: none
----------------------------------------------------------------------------
Q: What restocking fee applies to large items?
A: A restocking f

### Watch the out-of-scope question

There is nothing about parental leave in these three documents. A system without
grounding rules would happily invent a plausible-sounding policy — and a
confidently wrong HR answer is worse than no answer at all.

Ours should either refuse at the relevance floor, or return "I don't know" with
no citations.

## 12. A tiny evaluation

"It looked good in the demo" is not a quality bar. Even a handful of golden
questions catches regressions when you change chunk size, `k`, or the model.

We measure two things separately, because they fail for different reasons:

- **Retrieval hit rate** — did the right document reach the context?
- **Answer accuracy** — did the expected fact land in the answer?

Retrieval sets the ceiling. If the right chunk never arrives, no prompt tweak
can save you.

In [17]:
golden_set = [
    {"q": "How long are pickup orders held?",
     "doc": "Shipping Policy", "must_contain": "7"},
    {"q": "What is the return window for prepaid wireless phones?",
     "doc": "Returns Policy", "must_contain": "14"},
    {"q": "How many repairs before an item is replaced?",
     "doc": "Warranty Policy", "must_contain": "three"},
    {"q": "What does membership cost per year?",
     "doc": "Shipping Policy", "must_contain": "98"},
]

retrieval_hits = 0
answer_hits = 0

for case in golden_set:
    result = answer_question(case["q"], verbose=False)
    titles = {s["title"] for s in result["sources"]} or {
        h["title"] for h in retrieve(case["q"])
    }
    retrieved_ok = case["doc"] in titles
    answered_ok = case["must_contain"].lower() in result["answer"].lower()

    retrieval_hits += retrieved_ok
    answer_hits += answered_ok

    print(f"{'PASS' if answered_ok else 'FAIL'}  retrieval={'ok' if retrieved_ok else 'MISS'}"
          f"  | {case['q']}")
    print(f"      -> {result['answer'][:95]}")

total = len(golden_set)
print(f"\nRetrieval hit rate : {retrieval_hits}/{total}")
print(f"Answer accuracy    : {answer_hits}/{total}")

PASS  retrieval=ok  | How long are pickup orders held?
      -> Pickup orders are held for 7 days after the ready-for-pickup notification is sent [1].
PASS  retrieval=ok  | What is the return window for prepaid wireless phones?
      -> The return window for prepaid wireless phones is **14 days** [1].
PASS  retrieval=ok  | How many repairs before an item is replaced?
      -> Based on the extended plan, an item is replaced after **three** qualifying repairs [2].
PASS  retrieval=ok  | What does membership cost per year?
      -> Membership costs $98 per year [1].

Retrieval hit rate : 4/4
Answer accuracy    : 4/4


---
## 13. Where this simple pipeline breaks

Everything above is a *baseline*. It works, and it will disappoint you in
predictable ways. Each limitation maps to an enhancement in the Advanced RAG
section of the deck.

| Limitation | What it looks like | Enhancement |
| --- | --- | --- |
| Vocabulary mismatch | Exact codes like `POL-RET-004` are missed | Hybrid search (BM25 + vector) |
| Fixed `k` | Simple questions get noise, complex ones get starved | Adaptive or reranked retrieval |
| Chunk boundaries | A table split down the middle loses meaning | Structure-aware chunking |
| No query understanding | "it", "that one" fail with no chat history | Query rewriting |
| Single-hop only | Cannot combine facts across many documents | Multi-hop / agentic RAG |
| Similarity is not relevance | Top-4 by distance is not top-4 by usefulness | Cross-encoder reranking |
| Stale index | Edited a document? The index has no idea | Incremental re-indexing |

### Try breaking it yourself

Run the cell below. These are the failure modes, live. Seeing a system fail is a
faster education than being told it might.

In [18]:
stress_tests = [
    "What does POL-WAR-007 say?",                     # exact code lookup
    "What is the fee?",                               # ambiguous, no referent
    "Compare the return window for a TV and a drone", # multi-fact comparison
]

for question in stress_tests:
    answer_question(question)
    print("-" * 76)

Q: What does POL-WAR-007 say?
A: Based on the provided documents, POL-WAR-007 is the document ID for the "Warranty and Product Protection Policy," which is owned by Product Services and was last reviewed in January 2026 [1].
   grounded=True  refused=False  top_relevance=0.522
   sources: Warranty Policy (c015)
----------------------------------------------------------------------------
Q: What is the fee?
A: I don't know based on the available policy documents.
   grounded=False  refused=False  top_relevance=0.491
   sources: none
----------------------------------------------------------------------------
Q: Compare the return window for a TV and a drone
A: Based on the provided policy documents, the return window for consumer electronics (such as a TV) and drones is **30 days** [1].
   grounded=True  refused=False  top_relevance=0.475
   sources: Returns Policy (c010)
----------------------------------------------------------------------------


---
## Recap

You built a complete RAG system:

**Ingestion** — load, enrich metadata, chunk with overlap, embed, index
**Retrieval** — embed the query, nearest-neighbour search, relevance floor,
assemble citable context
**Generation** — a grounded prompt, a deterministic model, an LCEL chain ending
in `StrOutputParser`, and citation verification before the answer is returned

The parts worth carrying to production:

1. **Metadata from the start.** Citations and access control both depend on it.
2. **Let the system refuse.** "I don't know" is a valid, valuable answer.
3. **Verify what the model claims.** Check its citations point at real passages.
4. **Measure retrieval and generation separately.** They fail differently.
5. **Evaluate with a golden set.** Vibes do not survive a config change.